In [1]:
# Cell 1: Environment Setup, Target Chunks Extraction, and Golden Evaluation Set
from pathlib import Path
import sys
import numpy as np
import pandas as pd
from rich.console import Console
from rich.table import Table

console = Console()

# 1. Resolve workspace root dynamically
current_dir = Path.cwd().resolve()
project_root = current_dir.parent if current_dir.name == "notebooks" else current_dir
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.config.settings import settings
from src.chunking.text_splitter import ContextualTextSplitter
from src.ingestion.loader_factory import LoaderFactory
from src.preprocessing.cleaner import DocumentCleaner
from src.preprocessing.metadata_extractor import MetadataExtractor
from src.utils.file_utils import list_files_by_extension

# 2. Load the source document and produce fixed benchmark chunks
raw_dir = (project_root / "data" / "raw").resolve()
candidate_files = list_files_by_extension(raw_dir, ["pdf", "docx"])
assert len(candidate_files) > 0, f"No documents found in {raw_dir}"

raw_doc = LoaderFactory.load_document(candidate_files[0])
cleaned_doc = DocumentCleaner.clean_document(raw_doc)
target_doc = MetadataExtractor.enrich_document_metadata(cleaned_doc)

splitter = ContextualTextSplitter()
benchmark_chunks = splitter.split_document(target_doc)

print(f"Loaded document: {target_doc.metadata.file_name}")
print(f"Total benchmark chunks generated: {len(benchmark_chunks)}")

# 3. Small labeled query evaluation set (English + Multilingual/Arabic probes)
EVAL_QUERIES = [
    {
        "query": "What are the rules regarding annual leave entitlement and vacation days?",
        "lang": "en",
        "expected_keywords": ["leave", "annual", "days", "vacation"]
    },
    {
        "query": "What are the official working hours, probation period, or attendance policy?",
        "lang": "en",
        "expected_keywords": ["hours", "probation", "working", "attendance", "time"]
    },
    {
        "query": "ما هي شروط وسياسات الإجازات السنوية وساعات العمل للموظفين؟",
        "lang": "ar",
        "expected_keywords": ["leave", "vacation", "hours", "policy", "إجازة", "عمل"]
    }
]

print(f"Prepared {len(EVAL_QUERIES)} evaluation test queries.")

Loaded document: HR Policy.pdf
Total benchmark chunks generated: 45
Prepared 3 evaluation test queries.


In [2]:
# Cell 2: Multi-Model Embedding Comparison, Dimension & L2-Normalization Audit
import time
from sentence_transformers import SentenceTransformer

# النماذج المستهدفة بالمقارنة: نموذج المشروع الإنجليزي ونموذج متعدد اللغات للسياسات العربية/الإنجليزية
CANDIDATE_MODELS = {
    "bge-small-en": "BAAI/bge-small-en-v1.5",
    "multilingual-minilm": "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
}

chunk_texts = [c.content for c in benchmark_chunks]
model_audit_records = []
embeddings_store = {}

for model_key, model_name in CANDIDATE_MODELS.items():
    print(f"Loading and benchmarking model: {model_name}...")
    t0 = time.time()
    st_model = SentenceTransformer(model_name, device="cpu")
    load_time = time.time() - t0
    
    # استخراج التضمينات مع التطبيع
    t1 = time.time()
    embeddings = st_model.encode(
        sentences=chunk_texts,
        batch_size=16,
        show_progress_bar=False,
        normalize_embeddings=True,
        convert_to_numpy=True
    ).astype(np.float32)
    encode_time = time.time() - t1
    
    embeddings_store[model_key] = embeddings
    
    # الفحوصات الرياضية
    has_nans = bool(np.isnan(embeddings).any())
    has_infs = bool(np.isinf(embeddings).any())
    # فحص أن طول كل متجه = 1.0 (L2 Norm)
    norms = np.linalg.norm(embeddings, axis=1)
    is_l2_normalized = bool(np.allclose(norms, 1.0, atol=1e-4))
    
    model_audit_records.append({
        "Model Key": model_key,
        "HuggingFace Path": model_name,
        "Dimension": embeddings.shape[1],
        "Chunks Encoded": embeddings.shape[0],
        "Encode Time (s)": round(encode_time, 3),
        "Chars/Sec": int(sum(len(t) for t in chunk_texts) / max(0.001, encode_time)),
        "No NaNs/Infs": not (has_nans or has_infs),
        "Is L2 Normalized": is_l2_normalized
    })

audit_df = pd.DataFrame(model_audit_records)
display(audit_df)

Loading and benchmarking model: BAAI/bge-small-en-v1.5...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

The `sentences` argument was renamed and is now deprecated. Please use `inputs` instead.


Loading and benchmarking model: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2...


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 9.08MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

,Model Key,HuggingFace Path,Dimension,Chunks Encoded,Encode Time (s),Chars/Sec,No NaNs/Infs,Is L2 Normalized
0,bge-small-en,BAAI/bge-small-en-v1.5,384,45,1.261,14936,True,True
1,multilingual-minilm,sentence-transformers/paraphrase-multilingual-...,384,45,1.203,15658,True,True


In [3]:
# Cell 3: Cosine Similarity vs. Inner Product Equivalence Proof
from sklearn.metrics.pairwise import cosine_similarity

# نختبر على متجهات نموذج BGE المستخرجة في الخلية السابقة
bge_vectors = embeddings_store["bge-small-en"]
sample_vectors = bge_vectors[:min(5, len(bge_vectors))]

# 1. حساب التشابه باستخدام دالة Cosine Similarity الصريحة
explicit_cosine = cosine_similarity(sample_vectors, sample_vectors)

# 2. حساب التشابه باستخدام الضرب الداخلي المباشر (Inner Product / Dot Product)
inner_product = np.dot(sample_vectors, sample_vectors.T)

# 3. حساب الفرق المطلق بين المصفوفتين
diff_matrix = np.abs(explicit_cosine - inner_product)
max_discrepancy = float(np.max(diff_matrix))
are_identical = bool(np.allclose(explicit_cosine, inner_product, atol=1e-5))

# عرض النتائج في جدول مقارنة
table = Table(title="Metric Comparison: Cosine Similarity vs. Inner Product (Normalized Vectors)")
table.add_column("Property", style="cyan")
table.add_column("Value", style="green" if are_identical else "red")

table.add_row("Vectors Tested", str(len(sample_vectors)))
table.add_row("Vector L2 Norms", f"{np.linalg.norm(sample_vectors[0]):.6f}")
table.add_row("Max Absolute Discrepancy", f"{max_discrepancy:.2e}")
table.add_row("Mathematically Equivalent (<1e-5)", str(are_identical))
table.add_row("Sample Cosine Value [0, 0]", f"{explicit_cosine[0, 0]:.4f}")
table.add_row("Sample Inner Product [0, 0]", f"{inner_product[0, 0]:.4f}")

console.print(table)

 Metric Comparison: Cosine Similarity vs. Inner 
          Product (Normalized Vectors)          
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━┓
┃ Property                          ┃ Value    ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━┩
│ Vectors Tested                    │ 5        │
│ Vector L2 Norms                   │ 1.000000 │
│ Max Absolute Discrepancy          │ 2.38e-07 │
│ Mathematically Equivalent (<1e-5) │ True     │
│ Sample Cosine Value [0, 0]        │ 1.0000   │
│ Sample Inner Product [0, 0]       │ 1.0000   │
└───────────────────────────────────┴──────────┘

In [4]:
# Cell 4: FAISS Exact-Search Baseline, Persistence & Reload Verification
import tempfile
from src.vector_store.faiss_store import FaissVectorStore

# 1. تهيئة الفهرس الأساسي واختيار متجهات نموذج BGE
bge_vectors = embeddings_store["bge-small-en"]
vector_dim = bge_vectors.shape[1]

original_store = FaissVectorStore(dimension=vector_dim)
original_store.add_chunks(benchmark_chunks, bge_vectors)

test_query_vector = bge_vectors[0]  # نستخدم أول مقطع كاستعلام للتحقق التام

with tempfile.TemporaryDirectory() as tmp_dir:
    tmp_path = Path(tmp_dir)
    idx_file = tmp_path / "test_faiss.bin"
    meta_file = tmp_path / "test_meta.json"
    
    # 2. حفظ الفهرس والبيانات الوصفية
    original_store.save(idx_file, meta_file)
    
    idx_size_kb = idx_file.stat().st_size / 1024
    meta_size_kb = meta_file.stat().st_size / 1024
    
    # 3. إعادة تحميل الفهرس في كائن مستقل
    reloaded_store = FaissVectorStore(dimension=vector_dim)
    reloaded_store.load(idx_file, meta_file)
    
    # 4. مقارنة نتائج البحث بين الفهرس الأصلي والمعاد تحميله
    res_orig = original_store.search(test_query_vector, top_k=3)
    res_reload = reloaded_store.search(test_query_vector, top_k=3)
    
    # التحقق من تطابق المعرفات والدرجات
    orig_ids = [c.chunk_id for c, _ in res_orig]
    reload_ids = [c.chunk_id for c, _ in res_reload]
    orig_scores = [s for _, s in res_orig]
    reload_scores = [s for _, s in res_reload]
    
    ids_identical = orig_ids == reload_ids
    scores_identical = np.allclose(orig_scores, reload_scores, atol=1e-6)
    vector_counts_match = original_store.total_vectors() == reloaded_store.total_vectors()

# عرض جدول نتائج الاستمرارية والحفظ
table = Table(title="FAISS Persistence & Reload Verification")
table.add_column("Verification Step", style="cyan")
table.add_column("Value / Status", style="green" if (ids_identical and scores_identical) else "red")

table.add_row("Total Indexed Vectors", str(original_store.total_vectors()))
table.add_row("Index Binary Disk Size", f"{idx_size_kb:.2f} KB")
table.add_row("Metadata JSON Disk Size", f"{meta_size_kb:.2f} KB")
table.add_row("Vectors Count After Reload", str(reloaded_store.total_vectors()))
table.add_row("Chunk IDs Match Exactly", str(ids_identical))
table.add_row("Scores Match (tol < 1e-6)", str(scores_identical))
table.add_row("Top Match Score", f"{reload_scores[0]:.6f}")

console.print(table)

    FAISS Persistence & Reload Verification    
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┓
┃ Verification Step          ┃ Value / Status ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━┩
│ Total Indexed Vectors      │ 45             │
│ Index Binary Disk Size     │ 67.54 KB       │
│ Metadata JSON Disk Size    │ 28.82 KB       │
│ Vectors Count After Reload │ 45             │
│ Chunk IDs Match Exactly    │ True           │
│ Scores Match (tol < 1e-6)  │ True           │
│ Top Match Score            │ 1.000000       │
└────────────────────────────┴────────────────┘

In [6]:
# Cell 5: Stress-Testing Edge Cases (Empty Index, Dimension Mismatch & Metadata Integrity)

edge_case_records = []

# 1. اختبار الفهرس الفارغ
empty_store = FaissVectorStore(dimension=384)
try:
    dummy_query = np.zeros((384,), dtype=np.float32)
    empty_results = empty_store.search(dummy_query, top_k=5)
    passed_empty = (empty_results == [])
    edge_case_records.append({
        "Condition Tested": "Empty Index Search",
        "Expected Behavior": "Return empty list [] gracefully",
        "Passed": passed_empty,
        "Details": f"Returned: {empty_results}"
    })
except Exception as ex:
    edge_case_records.append({
        "Condition Tested": "Empty Index Search",
        "Expected Behavior": "Return empty list [] gracefully",
        "Passed": False,
        "Details": f"Threw Exception: {type(ex).__name__}"
    })

# 2. اختبار تمرير متجهات بأبعاد خاطئة (مثلاً 768 أو 512 بدلاً من 384)
invalid_dim_store = FaissVectorStore(dimension=384)
dummy_chunk = benchmark_chunks[0]
invalid_dim_vector = np.random.randn(1, 512).astype(np.float32)

try:
    invalid_dim_store.add_chunks([dummy_chunk], invalid_dim_vector)
    edge_case_records.append({
        "Condition Tested": "Mismatched Vector Dimension",
        "Expected Behavior": "Raise ValueError on dimension mismatch",
        "Passed": False,
        "Details": "Failed to catch mismatched dimension"
    })
except ValueError as ve:
    edge_case_records.append({
        "Condition Tested": "Mismatched Vector Dimension",
        "Expected Behavior": "Raise ValueError on dimension mismatch",
        "Passed": True,
        "Details": f"Correctly caught: {str(ve)[:55]}..."
    })
except Exception as ex:
    edge_case_records.append({
        "Condition Tested": "Mismatched Vector Dimension",
        "Expected Behavior": "Raise ValueError on dimension mismatch",
        "Passed": False,
        "Details": f"Unexpected error type: {type(ex).__name__}"
    })

# 3. اختبار تكامل البيانات الوصفية (Metadata-to-Vector Mapping Integrity)
mapping_clean = True
mapping_errors = []

for idx, chunk in enumerate(benchmark_chunks):
    stored_payload = original_store.metadata_store.get(idx)
    if not stored_payload:
        mapping_clean = False
        mapping_errors.append(f"Missing payload for index {idx}")
        break
    if stored_payload.get("chunk_id") != chunk.chunk_id:
        mapping_clean = False
        mapping_errors.append(f"ID mismatch at index {idx}: {stored_payload.get('chunk_id')} vs {chunk.chunk_id}")
        break
    if "content" not in stored_payload or len(stored_payload["content"]) == 0:
        mapping_clean = False
        mapping_errors.append(f"Empty content at index {idx}")
        break

edge_case_records.append({
    "Condition Tested": "Metadata-to-Vector Mapping",
    "Expected Behavior": "100% 1-to-1 sync between FAISS slot and metadata payload",
    "Passed": mapping_clean,
    "Details": "All chunk IDs, texts, and attributes aligned" if mapping_clean else "; ".join(mapping_errors)
})

edge_df = pd.DataFrame(edge_case_records)
display(edge_df)

,Condition Tested,Expected Behavior,Passed,Details
0,Empty Index Search,Return empty list [] gracefully,True,Returned: []
1,Mismatched Vector Dimension,Raise ValueError on dimension mismatch,True,Correctly caught: Vector dimension mismatch: e...
2,Metadata-to-Vector Mapping,100% 1-to-1 sync between FAISS slot and metada...,True,"All chunk IDs, texts, and attributes aligned"


In [7]:
# Cell 6: Labeled Query Retrieval Benchmarking (BGE vs. Multilingual)
from sentence_transformers import SentenceTransformer

retrieval_eval_records = []

# بناء فهرسين مستقلين للاختبار
stores = {}
for m_key, vectors in embeddings_store.items():
    s = FaissVectorStore(dimension=vectors.shape[1])
    s.add_chunks(benchmark_chunks, vectors)
    stores[m_key] = s

# تحميل كائنات النماذج لترميز الاستعلامات
loaded_models = {
    m_key: SentenceTransformer(m_path, device="cpu")
    for m_key, m_path in CANDIDATE_MODELS.items()
}

for q_item in EVAL_QUERIES:
    query_text = q_item["query"]
    q_lang = q_item["lang"]
    exp_keywords = q_item["expected_keywords"]
    
    for m_key, store in stores.items():
        st_mod = loaded_models[m_key]
        
        # إضافة بادئة BGE فقط إذا كان النموذج هو BGE
        q_encoded_text = (
            f"Represent this sentence for searching relevant passages: {query_text}"
            if "bge" in m_key
            else query_text
        )
        
        q_vec = st_mod.encode(
            q_encoded_text, 
            normalize_embeddings=True, 
            convert_to_numpy=True
        ).astype(np.float32)
        
        hits = store.search(q_vec, top_k=3)
        
        # التحقق من وجود الكلمات المفتاحية في المقاطع المسترجعة
        combined_text = " ".join([c.content.lower() for c, _ in hits])
        matched_kws = [kw for kw in exp_keywords if kw.lower() in combined_text]
        hit_success = len(matched_kws) > 0
        top_score = hits[0][1] if hits else 0.0
        
        retrieval_eval_records.append({
            "Query Language": q_lang.upper(),
            "Model": m_key,
            "Query Preview": query_text[:40] + "...",
            "Top-1 Score": round(float(top_score), 4),
            "Matched Keywords": f"{len(matched_kws)}/{len(exp_keywords)}",
            "Hit@3": hit_success
        })

retrieval_df = pd.DataFrame(retrieval_eval_records)
display(retrieval_df)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

,Query Language,Model,Query Preview,Top-1 Score,Matched Keywords,Hit@3
0,EN,bge-small-en,What are the rules regarding annual leav...,0.7304,3/4,True
1,EN,multilingual-minilm,What are the rules regarding annual leav...,0.6029,3/4,True
2,EN,bge-small-en,"What are the official working hours, pro...",0.6371,3/5,True
3,EN,multilingual-minilm,"What are the official working hours, pro...",0.5422,2/5,True
4,AR,bge-small-en,ما هي شروط وسياسات الإجازات السنوية وساع...,0.5634,1/6,True
5,AR,multilingual-minilm,ما هي شروط وسياسات الإجازات السنوية وساع...,0.6403,4/6,True


In [8]:
# Cell 7: Final Metrics Summary & Production Architectural Decisions

summary_table = Table(title="Notebook 04: Embeddings & Vector Index Experimentation Summary")
summary_table.add_column("Experimental Dimension", style="cyan")
summary_table.add_column("Empirical Finding / Benchmark", style="green")

summary_table.add_row("Base Dense Dimension", f"{vector_dim} (Float32)")
summary_table.add_row("Vector Normalization", "Strict L2-Norm (||v|| = 1.0000)")
summary_table.add_row("Metric Equivalence", "Cosine Similarity == Inner Product (Delta < 1e-5)")
summary_table.add_row("Index Reload Verification", "100% Deterministic (IDs & Scores Matched)")
summary_table.add_row("Edge Case Tolerance", "Handled (Empty Index, Dimension Check, Metadata Sync)")
summary_table.add_row("Primary Model (English)", "BAAI/bge-small-en-v1.5 (High In-Domain Precision)")
summary_table.add_row("Multilingual Fallback", "paraphrase-multilingual-MiniLM-L12-v2 (Zero-code change)")

console.print(summary_table)

architectural_decision = """
[bold yellow]================ PRODUCTION ARCHITECTURAL DECISIONS ================[/bold yellow]

1. EMBEDDING MODEL:
   - Primary: 'BAAI/bge-small-en-v1.5' (384-d). Excellent semantic discrimination for HR policies.
   - Multilingual Capability: If enterprise HR queries are in Arabic, switch EMBEDDING_MODEL in .env to
     'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2' (shares identical 384 dimension).

2. SIMILARITY METRIC & FAISS INDEX TYPE:
   - Index Type: faiss.IndexFlatIP (Exact Exhaustive Search via Inner Product).
   - Rationale: HR policy corpora range from dozens to thousands of chunks; exact search gives 100% recall
     without indexing latency penalties, cluster training overhead (IVF), or index degradation (HNSW).

3. NORMALIZATION POLICY:
   - Mandatory L2 normalization on all document and query vectors (normalize_embeddings=True).
   - Under unit-norm vectors, Inner Product is mathematically identical to Cosine Similarity,
     maximizing search throughput in FAISS.

4. PERSISTENCE & METADATA CONTRACT:
   - Persist binaries via faiss.write_index ('faiss_index.bin').
   - Keep sidecar chunk payload dictionary ('chunks_metadata.json') indexed strictly by sequential FAISS slot IDs.
"""
console.print(architectural_decision)

             Notebook 04: Embeddings & Vector Index Experimentation Summary             
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Experimental Dimension    ┃ Empirical Finding / Benchmark                            ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ Base Dense Dimension      │ 384 (Float32)                                            │
│ Vector Normalization      │ Strict L2-Norm (||v|| = 1.0000)                          │
│ Metric Equivalence        │ Cosine Similarity == Inner Product (Delta < 1e-5)        │
│ Index Reload Verification │ 100% Deterministic (IDs & Scores Matched)                │
│ Edge Case Tolerance       │ Handled (Empty Index, Dimension Check, Metadata Sync)    │
│ Primary Model (English)   │ BAAI/bge-small-en-v1.5 (High In-Domain Precision)        │
│ Multilingual Fallback     │ paraphrase-multilingual-MiniLM-L12-v2 (Zero-code change) │
└───────────────────────────┴──────────────────────────────────────────────────────────┘

================ PRODUCTION ARCHITECTURAL DECISIONS ================

1. EMBEDDING MODEL:
   - Primary: 'BAAI/bge-small-en-v1.5' (384-d). Excellent semantic discrimination for HR policies.
   - Multilingual Capability: If enterprise HR queries are in Arabic, switch EMBEDDING_MODEL in .env to
     'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2' (shares identical 384 dimension).

2. SIMILARITY METRIC & FAISS INDEX TYPE:
   - Index Type: faiss.IndexFlatIP (Exact Exhaustive Search via Inner Product).
   - Rationale: HR policy corpora range from dozens to thousands of chunks; exact search gives 100% recall
     without indexing latency penalties, cluster training overhead (IVF), or index degradation (HNSW).

3. NORMALIZATION POLICY:
   - Mandatory L2 normalization on all document and query vectors (normalize_embeddings=True).
   - Under unit-norm vectors, Inner Product is mathematically identical to Cosine Similarity,
     maximizing search throughput in FAISS.

4. PERSISTENCE & METADATA CONTRACT:
   - Persist binaries via faiss.write_index ('faiss_index.bin').
   - Keep sidecar chunk payload dictionary ('chunks_metadata.json') indexed strictly by sequential FAISS slot IDs.